In [1]:

# ============================================================
# CELL 1 — PERSISTENT PROJECT SETUP + INSTALLS
# ============================================================

from google.colab import drive
import os
import sys
import subprocess
import shutil

# ------------------------------------------------------------
# FORCE CLEAN GOOGLE DRIVE MOUNT POINT
# ------------------------------------------------------------

MOUNT_POINT = "/content/drive"

if os.path.ismount(MOUNT_POINT):
    print("✓ Google Drive already mounted.")
else:
    if os.path.exists(MOUNT_POINT):
        print("🧹 Cleaning old /content/drive mount point...")
        shutil.rmtree(MOUNT_POINT)

    os.makedirs(MOUNT_POINT, exist_ok=True)

    print("🔗 Mounting Google Drive...")
    drive.mount(MOUNT_POINT)

# ------------------------------------------------------------
# INSTALL PACKAGES
# ------------------------------------------------------------

print("📦 Installing required packages...")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "piper-tts==1.5.0",
    "openai-whisper",
    "moviepy"
])

# ------------------------------------------------------------
# FFmpeg
# ------------------------------------------------------------

print("🎬 Checking FFmpeg...")

ffmpeg_check = subprocess.run(
    ["ffmpeg", "-version"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

if ffmpeg_check.returncode != 0:
    print("⬇️ Installing FFmpeg...")

    subprocess.check_call([
        "apt-get",
        "update",
        "-qq"
    ])

    subprocess.check_call([
        "apt-get",
        "install",
        "-y",
        "-qq",
        "ffmpeg"
    ])

# ------------------------------------------------------------
# PERSISTENT PROJECT
# ------------------------------------------------------------

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "Sc_Gamecocks_Video_Pipeline"
)

DIRECTORIES = {
    "models": os.path.join(PROJECT_DIR, "models"),
    "piper": os.path.join(PROJECT_DIR, "models", "piper"),
    "whisper": os.path.join(PROJECT_DIR, "models", "whisper"),
    "voices": os.path.join(PROJECT_DIR, "voices"),
    "media": os.path.join(PROJECT_DIR, "media"),
    "music": os.path.join(PROJECT_DIR, "music"),
    "outputs": os.path.join(PROJECT_DIR, "outputs"),
    "temp": os.path.join(PROJECT_DIR, "temp"),
    "config": os.path.join(PROJECT_DIR, "config"),
}

for path in DIRECTORIES.values():
    os.makedirs(path, exist_ok=True)

# ------------------------------------------------------------
# RUNTIME DIRECTORIES
# ------------------------------------------------------------

RUNTIME_DIR = "/content/sc_pipeline"

RUNTIME_DIRECTORIES = {
    "models": os.path.join(RUNTIME_DIR, "models"),
    "voices": os.path.join(RUNTIME_DIR, "voices"),
    "media": os.path.join(RUNTIME_DIR, "media"),
    "music": os.path.join(RUNTIME_DIR, "music"),
    "outputs": os.path.join(RUNTIME_DIR, "outputs"),
    "temp": os.path.join(RUNTIME_DIR, "temp"),
}

for path in RUNTIME_DIRECTORIES.values():
    os.makedirs(path, exist_ok=True)

# ------------------------------------------------------------
# RUNTIME PATHS
# ------------------------------------------------------------

VOICE_DIR = "/content/voices"
MEDIA_DIR = RUNTIME_DIRECTORIES["media"]
MUSIC_DIR = RUNTIME_DIRECTORIES["music"]
OUTPUT_DIR = RUNTIME_DIRECTORIES["outputs"]
TEMP_DIR = RUNTIME_DIRECTORIES["temp"]

os.makedirs(VOICE_DIR, exist_ok=True)

PERSISTENT_VOICE_DIR = DIRECTORIES["voices"]
PERSISTENT_WHISPER_DIR = DIRECTORIES["whisper"]
PERSISTENT_MEDIA_DIR = DIRECTORIES["media"]
PERSISTENT_MUSIC_DIR = DIRECTORIES["music"]
PERSISTENT_OUTPUT_DIR = DIRECTORIES["outputs"]

# ------------------------------------------------------------
# COMPLETE
# ------------------------------------------------------------

print("\n========================================")
print("🎬 GAMECOCKS VIDEO PIPELINE")
print("========================================")
print("✅ Google Drive mounted")
print("✅ Piper installed")
print("✅ Whisper installed")
print("✅ MoviePy installed")
print("✅ FFmpeg available")
print("📁 Project:", PROJECT_DIR)

print("\n========================================")
print("✅ CELL 1 COMPLETE")
print("========================================")

✓ Google Drive already mounted.
📦 Installing required packages...
🎬 Checking FFmpeg...

🎬 GAMECOCKS VIDEO PIPELINE
✅ Google Drive mounted
✅ Piper installed
✅ Whisper installed
✅ MoviePy installed
✅ FFmpeg available
📁 Project: /content/drive/MyDrive/Sc_Gamecocks_Video_Pipeline

✅ CELL 1 COMPLETE


In [2]:

# ============================================================
# CELL 2 — PERSISTENT PIPER VOICES
# ============================================================

import os
import shutil
import urllib.request

# Persistent + runtime voice folders
VOICE_DIR = RUNTIME_DIRECTORIES["voices"]
RUNTIME_VOICE_DIR = "/content/voices"

os.makedirs(VOICE_DIR, exist_ok=True)
os.makedirs(RUNTIME_VOICE_DIR, exist_ok=True)

# Piper voices we want available
PIPER_VOICE_URLS = {
    "Lessac": "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx",
    "Ryan": "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/ryan/medium/en_US-ryan-medium.onnx",
}

def download_voice(name, url):
    filename = os.path.basename(url)
    persistent_path = os.path.join(VOICE_DIR, filename)
    runtime_path = os.path.join(RUNTIME_VOICE_DIR, filename)

    # Download to persistent Drive if missing
    if not os.path.exists(persistent_path):
        print(f"Downloading Piper voice: {name}")
        urllib.request.urlretrieve(url, persistent_path)

    # Copy to runtime
    if not os.path.exists(runtime_path):
        shutil.copy2(persistent_path, runtime_path)

    return runtime_path

# Download/copy all voices
VOICE_PATHS = {}

for name, url in PIPER_VOICE_URLS.items():
    try:
        VOICE_PATHS[name] = download_voice(name, url)
        print(f"✓ {name}: {VOICE_PATHS[name]}")
    except Exception as e:
        print(f"✗ Could not load {name}: {e}")

# Compatibility with the rest of the notebook
if "Lessac" in VOICE_PATHS:
    VOICE = VOICE_PATHS["Lessac"]
else:
    available = list(VOICE_PATHS.values())
    if available:
        VOICE = available[0]
    else:
        raise RuntimeError("No Piper voices could be loaded.")

print("\nAvailable Piper voices:")
for name, path in VOICE_PATHS.items():
    print(f"  • {name} → {path}")

print(f"\nDefault voice: {VOICE}")

✓ Lessac: /content/voices/en_US-lessac-medium.onnx
✓ Ryan: /content/voices/en_US-ryan-medium.onnx

Available Piper voices:
  • Lessac → /content/voices/en_US-lessac-medium.onnx
  • Ryan → /content/voices/en_US-ryan-medium.onnx

Default voice: /content/voices/en_US-lessac-medium.onnx


In [ ]:

# ============================================================
# CELL 3 — PERSISTENT WHISPER MODEL
# ============================================================

import os
import shutil
import whisper

PERSISTENT_WHISPER_DIR = "/content/drive/MyDrive/Sc_Gamecocks_Video_Pipeline/models/whisper"
RUNTIME_WHISPER_DIR = "/content/whisper_models"

os.makedirs(PERSISTENT_WHISPER_DIR, exist_ok=True)
os.makedirs(RUNTIME_WHISPER_DIR, exist_ok=True)

WHISPER_MODEL_NAME = "base"

persistent_model = os.path.join(
    PERSISTENT_WHISPER_DIR,
    f"{WHISPER_MODEL_NAME}.pt"
)

runtime_model = os.path.join(
    RUNTIME_WHISPER_DIR,
    f"{WHISPER_MODEL_NAME}.pt"
)

# Download only if the model is not already on Drive
if not os.path.exists(persistent_model):

    print("⬇️ First-time Whisper download...")

    model = whisper.load_model(
        WHISPER_MODEL_NAME,
        download_root=PERSISTENT_WHISPER_DIR
    )

    # Find downloaded model
    downloaded = os.path.join(
        PERSISTENT_WHISPER_DIR,
        "base.pt"
    )

    if os.path.exists(downloaded):
        persistent_model = downloaded

else:
    print("✓ Whisper model already stored on Drive.")

# Copy model to fast local runtime storage
if not os.path.exists(runtime_model):
    shutil.copy2(
        persistent_model,
        runtime_model
    )

# Load Whisper from local runtime storage
print("🎙️ Loading Whisper...")

whisper_model = whisper.load_model(
    WHISPER_MODEL_NAME,
    download_root=RUNTIME_WHISPER_DIR
)

print("\n========================================")
print("✅ WHISPER READY")
print("========================================")
print("Model:", WHISPER_MODEL_NAME)
print("Persistent:", persistent_model)

✓ Whisper model already stored on Drive.
🎙️ Loading Whisper...

✅ WHISPER READY
Model: base
Persistent: /content/drive/MyDrive/Sc_Gamecocks_Video_Pipeline/models/whisper/base.pt


In [ ]:

# ============================================================
# CELL 4 — MEDIA UPLOAD & PERSISTENT STORAGE
# ============================================================

from google.colab import files
import os
import shutil

# Persistent media folder from Cell 1
PERSISTENT_MEDIA_DIR = (
    "/content/drive/MyDrive/"
    "Sc_Gamecocks_Video_Pipeline/media"
)

# Fast local runtime folder
MEDIA_DIR = "/content/sc_pipeline/media"

os.makedirs(PERSISTENT_MEDIA_DIR, exist_ok=True)
os.makedirs(MEDIA_DIR, exist_ok=True)

# ============================================================
# UPLOAD MEDIA
# ============================================================

print("📤 Upload your images or ONE video.")
print("You can select multiple images at once.\n")

uploaded = files.upload()

# ============================================================
# COPY TO BOTH DRIVE + LOCAL RUNTIME
# ============================================================

MEDIA_FILES = []

for filename in uploaded.keys():

    source = os.path.join("/content", filename)

    # Persistent copy
    drive_destination = os.path.join(
        PERSISTENT_MEDIA_DIR,
        filename
    )

    shutil.copy2(
        source,
        drive_destination
    )

    # Fast runtime copy
    runtime_destination = os.path.join(
        MEDIA_DIR,
        filename
    )

    shutil.copy2(
        source,
        runtime_destination
    )

    MEDIA_FILES.append(runtime_destination)

    print(f"✓ {filename}")

# ============================================================
# CHECK MEDIA TYPE
# ============================================================

IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
)

VIDEO_EXTENSIONS = (
    ".mp4",
    ".mov",
    ".mkv",
    ".avi",
    ".webm"
)

image_files = [
    f for f in MEDIA_FILES
    if f.lower().endswith(IMAGE_EXTENSIONS)
]

video_files = [
    f for f in MEDIA_FILES
    if f.lower().endswith(VIDEO_EXTENSIONS)
]

if len(video_files) > 1:
    raise RuntimeError(
        "Please upload only ONE video at a time."
    )

print("\n========================================")
print("✅ MEDIA READY")
print("========================================")
print(f"Images: {len(image_files)}")
print(f"Videos: {len(video_files)}")
print("Persistent folder:", PERSISTENT_MEDIA_DIR)

📤 Upload your images or ONE video.
You can select multiple images at once.



Saving images - 2026-09-11T124401.313.jpeg to images - 2026-09-11T124401.313.jpeg
Saving images - 2026-09-11T124440.164.jpeg to images - 2026-09-11T124440.164.jpeg
Saving images - 2026-09-11T124326.578.jpeg to images - 2026-09-11T124326.578.jpeg
Saving images - 2026-09-11T124237.943.jpeg to images - 2026-09-11T124237.943.jpeg
✓ images - 2026-09-11T124401.313.jpeg
✓ images - 2026-09-11T124440.164.jpeg
✓ images - 2026-09-11T124326.578.jpeg
✓ images - 2026-09-11T124237.943.jpeg

✅ MEDIA READY
Images: 4
Videos: 0
Persistent folder: /content/drive/MyDrive/Sc_Gamecocks_Video_Pipeline/media


In [ ]:

# ============================================================
# CELL 5 — SCRIPT & VIDEO SETTINGS
# ============================================================

SCRIPT = """
Brooke Quarles Daniels is already putting in the work.

The South Carolina newcomer is preparing for what could be an important season with the Gamecocks.

And if this training is any indication, she is ready for the challenge.
"""

# ============================================================
# VIDEO FORMAT
# ============================================================

VIDEO_WIDTH = 1080
VIDEO_HEIGHT = 1920

VIDEO_FORMAT = "9:16"

# ============================================================
# NARRATION
# ============================================================

VOICE_NAME = "Lessac - Male"

SPEECH_RATE = 0.98

# ============================================================
# CAPTIONS
# ============================================================

ADD_CAPTIONS = True

CAPTION_FONT_SIZE = 58

# ============================================================
# HOOK
# ============================================================

USE_HOOK = True

HOOK_TEXT = "BROOKE QUARLES DANIELS"
HOOK_SUBTEXT = "IS PUTTING IN THE WORK"

HOOK_DURATION = 1.5

# ============================================================
# AUDIO
# ============================================================

ADD_MUSIC = True

MUSIC_VOLUME = 0.12

MUTE_ORIGINAL_AUDIO = True

# ============================================================
# OUTPUT
# ============================================================

OUTPUT = (
    "/content/drive/MyDrive/"
    "Sc_Gamecocks_Video_Pipeline/outputs/"
    "gamecocks_story.mp4"
)

print("========================================")
print("🎬 VIDEO SETTINGS READY")
print("========================================")

print("Format:", VIDEO_FORMAT)
print("Resolution:", f"{VIDEO_WIDTH}x{VIDEO_HEIGHT}")
print("Voice:", VOICE_NAME)
print("Speech rate:", SPEECH_RATE)
print("Captions:", ADD_CAPTIONS)
print("Hook:", USE_HOOK)
print("Music:", ADD_MUSIC)
print("Output:", OUTPUT)

🎬 VIDEO SETTINGS READY
Format: 9:16
Resolution: 1080x1920
Voice: Lessac - Male
Speech rate: 0.98
Captions: True
Hook: True
Music: True
Output: /content/drive/MyDrive/Sc_Gamecocks_Video_Pipeline/outputs/gamecocks_story.mp4


In [ ]:

# ============================================================
# CELL 6 — NATURAL PIPER NARRATION
# ============================================================

import os
import re
import subprocess
import wave

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

VOICE = "/content/voices/en_US-lessac-medium.onnx"

RAW_AUDIO = "/content/sc_pipeline/temp/narration_raw.wav"
AUDIO_FILE = "/content/narration.wav"

os.makedirs(
    "/content/sc_pipeline/temp",
    exist_ok=True
)

# ------------------------------------------------------------
# CHECK VOICE
# ------------------------------------------------------------

if not os.path.exists(VOICE):

    raise FileNotFoundError(
        f"Piper voice not found:\n{VOICE}\n\n"
        "Run Cell 2 first."
    )

# ------------------------------------------------------------
# SPEECH SETTINGS
# ------------------------------------------------------------

SPEECH_RATE = 0.98

SENTENCE_PAUSE = 0.18
PARAGRAPH_PAUSE = 0.40

# ------------------------------------------------------------
# CLEAN SCRIPT
# ------------------------------------------------------------

text = SCRIPT.replace("\r\n", "\n")
text = text.replace("\r", "\n")

paragraphs = [
    p.strip()
    for p in re.split(
        r"\n\s*\n",
        text
    )
    if p.strip()
]

# ------------------------------------------------------------
# CREATE NATURAL SPEECH CHUNKS
# ------------------------------------------------------------

speech_chunks = []

for paragraph in paragraphs:

    sentences = re.split(
        r"(?<=[.!?])\s+",
        paragraph
    )

    sentences = [
        s.strip()
        for s in sentences
        if s.strip()
    ]

    for sentence in sentences:
        speech_chunks.append(sentence)

# ------------------------------------------------------------
# GENERATE EACH SENTENCE
# ------------------------------------------------------------

sentence_files = []

print("========================================")
print("🎙️ NATURAL PIPER NARRATION")
print("========================================")

print("Voice:", "Lessac")
print("Speech rate:", SPEECH_RATE)
print("Sentence pause:", SENTENCE_PAUSE)
print("Paragraph pause:", PARAGRAPH_PAUSE)

for index, sentence in enumerate(speech_chunks):

    sentence_file = (
        f"/content/sc_pipeline/temp/"
        f"sentence_{index:03d}.wav"
    )

    print(
        f"🎙️ {index + 1}/"
        f"{len(speech_chunks)}: "
        f"{sentence}"
    )

    process = subprocess.run(
        [
            "piper",
            "--model",
            VOICE,
            "--length-scale",
            str(1.0 / SPEECH_RATE),
            "--output_file",
            sentence_file
        ],
        input=sentence.encode("utf-8"),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    if process.returncode != 0:

        print(
            process.stderr.decode(
                "utf-8",
                errors="ignore"
            )
        )

        raise RuntimeError(
            "Piper failed to generate narration."
        )

    sentence_files.append(sentence_file)

# ------------------------------------------------------------
# BUILD CONCAT LIST WITH REAL PAUSES
# ------------------------------------------------------------

concat_file = (
    "/content/sc_pipeline/temp/"
    "narration_concat.txt"
)

with open(
    concat_file,
    "w",
    encoding="utf-8"
) as f:

    for index, filename in enumerate(sentence_files):

        f.write(
            f"file '{filename}'\n"
        )

        # Add silence between sentences.
        if index < len(sentence_files) - 1:

            pause = (
                PARAGRAPH_PAUSE
                if (
                    index + 1 < len(speech_chunks)
                    and paragraphs
                )
                else SENTENCE_PAUSE
            )

            # Actual silence is added later with FFmpeg.
            # Keep sentence files separate here.

# ------------------------------------------------------------
# CREATE SILENCE FILES
# ------------------------------------------------------------

SENTENCE_SILENCE = (
    "/content/sc_pipeline/temp/"
    "sentence_silence.wav"
)

PARAGRAPH_SILENCE = (
    "/content/sc_pipeline/temp/"
    "paragraph_silence.wav"
)

for duration, output in [
    (SENTENCE_PAUSE, SENTENCE_SILENCE),
    (PARAGRAPH_PAUSE, PARAGRAPH_SILENCE)
]:

    result = subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-f", "lavfi",
            "-i",
            "anullsrc=r=48000:cl=mono",
            "-t",
            str(duration),
            "-c:a",
            "pcm_s16le",
            output
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE
    )

    if result.returncode != 0:

        print(
            result.stderr.decode(
                "utf-8",
                errors="ignore"
            )
        )

        raise RuntimeError(
            "Could not create silence."
        )

# ------------------------------------------------------------
# COMBINE SENTENCES
# ------------------------------------------------------------

inputs = []
filter_parts = []

for index, sentence_file in enumerate(sentence_files):

    inputs.extend([
        "-i",
        sentence_file
    ])

    filter_parts.append(
        f"[{index}:a]"
    )

    if index < len(sentence_files) - 1:

        inputs.extend([
            "-i",
            SENTENCE_SILENCE
        ])

        silence_index = len(inputs) // 2 - 1

        filter_parts.append(
            f"[{silence_index}:a]"
        )

filter_string = "".join(
    filter_parts
)

input_count = len(filter_parts)

concat_filter = (
    filter_string
    + f"concat=n={input_count}:v=0:a=1[out]"
)

command = [
    "ffmpeg",
    "-y"
]

command.extend(inputs)

command.extend([
    "-filter_complex",
    concat_filter,
    "-map",
    "[out]",
    "-ar",
    "48000",
    "-ac",
    "1",
    "-c:a",
    "pcm_s16le",
    RAW_AUDIO
])

result = subprocess.run(
    command,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

if result.returncode != 0:

    print(
        result.stderr.decode(
            "utf-8",
            errors="ignore"
        )
    )

    raise RuntimeError(
        "Could not combine narration."
    )

# ------------------------------------------------------------
# FINAL AUDIO POLISH
# ------------------------------------------------------------

print("\n🎚️ Polishing narration...")

audio_filter = (
    "highpass=f=70,"
    "lowpass=f=14000,"
    "acompressor="
    "threshold=0.08:"
    "ratio=2.5:"
    "attack=15:"
    "release=180:"
    "makeup=1,"
    "loudnorm="
    "I=-16:"
    "LRA=7:"
    "TP=-1.5"
)

result = subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        RAW_AUDIO,
        "-af",
        audio_filter,
        "-ar",
        "48000",
        "-ac",
        "1",
        "-c:a",
        "pcm_s16le",
        AUDIO_FILE
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

if result.returncode != 0:

    print(
        result.stderr.decode(
            "utf-8",
            errors="ignore"
        )
    )

    raise RuntimeError(
        "Narration audio processing failed."
    )

# ------------------------------------------------------------
# VALIDATE
# ------------------------------------------------------------

if not os.path.exists(AUDIO_FILE):

    raise RuntimeError(
        "Narration file was not created."
    )

with wave.open(
    AUDIO_FILE,
    "rb"
) as wf:

    duration = (
        wf.getnframes()
        / wf.getframerate()
    )

print("\n========================================")
print("✅ NATURAL NARRATION READY")
print("========================================")

print("Duration:", round(duration, 2), "seconds")
print("Audio:", AUDIO_FILE)

print("\n✓ Sentence pauses added")
print("✓ Compression applied")
print("✓ Loudness normalized")
print("✓ Ready for video pipeline")

🎙️ NATURAL PIPER NARRATION
Voice: Lessac
Speech rate: 0.98
Sentence pause: 0.18
Paragraph pause: 0.4
🎙️ 1/3: Brooke Quarles Daniels is already putting in the work.
🎙️ 2/3: The South Carolina newcomer is preparing for what could be an important season with the Gamecocks.
🎙️ 3/3: And if this training is any indication, she is ready for the challenge.

🎚️ Polishing narration...

✅ NATURAL NARRATION READY
Duration: 8.28 seconds
Audio: /content/narration.wav

✓ Sentence pauses added
✓ Compression applied
✓ Loudness normalized
✓ Ready for video pipeline


In [ ]:

# ============================================================
# CELL 7 — DISCOVER & ORGANIZE MEDIA
# ============================================================

import os

# Use the runtime media folder created by Cell 4
MEDIA_DIR = "/content/sc_pipeline/media"

if not os.path.exists(MEDIA_DIR):
    raise FileNotFoundError(
        f"Media folder not found:\n{MEDIA_DIR}\n"
        "Make sure Cell 4 has been run."
    )

# ============================================================
# SUPPORTED FILE TYPES
# ============================================================

IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
)

VIDEO_EXTENSIONS = (
    ".mp4",
    ".mov",
    ".mkv",
    ".avi",
    ".webm"
)

# ============================================================
# DISCOVER FILES
# ============================================================

MEDIA_FILES = []

for filename in sorted(os.listdir(MEDIA_DIR)):

    path = os.path.join(
        MEDIA_DIR,
        filename
    )

    if os.path.isfile(path):
        MEDIA_FILES.append(path)

# Separate images and videos
image_files = [
    f for f in MEDIA_FILES
    if f.lower().endswith(IMAGE_EXTENSIONS)
]

video_files = [
    f for f in MEDIA_FILES
    if f.lower().endswith(VIDEO_EXTENSIONS)
]

# ============================================================
# VALIDATION
# ============================================================

if not image_files and not video_files:
    raise RuntimeError(
        "No images or videos were found in the media folder."
    )

if len(video_files) > 1:
    raise RuntimeError(
        "Only ONE video can be used at a time."
    )

# ============================================================
# DISPLAY
# ============================================================

print("========================================")
print("🎞️ MEDIA DISCOVERY")
print("========================================")

print(f"\n📸 Images: {len(image_files)}")

for file in image_files:
    print("  ✓", os.path.basename(file))

print(f"\n🎥 Videos: {len(video_files)}")

for file in video_files:
    print("  ✓", os.path.basename(file))

print("\n========================================")
print("✅ MEDIA READY FOR RENDERING")
print("========================================")

🎞️ MEDIA DISCOVERY

📸 Images: 8
  ✓ 1787572852978.jpeg
  ✓ 1787572853068.jpeg
  ✓ Screenshot 2026-09-09 200717.png
  ✓ Screenshot 2026-09-09 200857.png
  ✓ images - 2026-09-11T124237.943.jpeg
  ✓ images - 2026-09-11T124326.578.jpeg
  ✓ images - 2026-09-11T124401.313.jpeg
  ✓ images - 2026-09-11T124440.164.jpeg

🎥 Videos: 0

✅ MEDIA READY FOR RENDERING


In [ ]:

# ============================================================
# CELL 8 — GET NARRATION DURATION
# ============================================================

import os
import subprocess

AUDIO_FILE = "/content/narration.wav"

if not os.path.exists(AUDIO_FILE):
    raise FileNotFoundError(
        "Narration file not found.\n"
        "Cell 6 must be run first."
    )

result = subprocess.run(
    [
        "ffprobe",
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        AUDIO_FILE
    ],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        "Could not determine narration duration."
    )

NARRATION_DURATION = float(result.stdout.strip())

print("========================================")
print("⏱️ NARRATION TIMING")
print("========================================")
print(
    f"Narration duration: "
    f"{NARRATION_DURATION:.2f} seconds"
)

print("========================================")
print("✅ TIMING READY")
print("========================================")

⏱️ NARRATION TIMING
Narration duration: 8.28 seconds
✅ TIMING READY


In [ ]:

# ============================================================
# CELL 9 — ANIMATED IMAGE CLIPS
# ============================================================

import os
import subprocess

VIDEO_WIDTH = 1080
VIDEO_HEIGHT = 1920

IMAGE_CLIPS_DIR = "/content/sc_pipeline/temp/image_clips"

os.makedirs(IMAGE_CLIPS_DIR, exist_ok=True)

if not image_files:
    print("ℹ️ No images found.")
    print("A video input will be handled by the next cell.")

else:

    # Spread narration time across all images
    image_duration = NARRATION_DURATION / len(image_files)

    print("========================================")
    print("🎬 CREATING IMAGE MOTION")
    print("========================================")

    print(f"Images: {len(image_files)}")
    print(f"Time per image: {image_duration:.2f}s")

    generated_clips = []

    for index, image in enumerate(image_files):

        output = os.path.join(
            IMAGE_CLIPS_DIR,
            f"image_{index:03d}.mp4"
        )

        # Alternate between zoom-in and zoom-out
        if index % 2 == 0:
            zoom_filter = (
                "zoompan="
                "z='min(zoom+0.0008,1.12)':"
                "x='iw/2-(iw/zoom/2)':"
                "y='ih/2-(ih/zoom/2)':"
                f"d=1:"
                f"s={VIDEO_WIDTH}x{VIDEO_HEIGHT}:"
                "fps=30"
            )
        else:
            zoom_filter = (
                "zoompan="
                "z='if(lte(zoom,1.001),1.12,max(zoom-0.0008,1.0))':"
                "x='iw/2-(iw/zoom/2)':"
                "y='ih/2-(ih/zoom/2)':"
                f"d=1:"
                f"s={VIDEO_WIDTH}x{VIDEO_HEIGHT}:"
                "fps=30"
            )

        command = [
            "ffmpeg",
            "-y",
            "-loop", "1",
            "-i", image,
            "-vf",
            (
                "scale=1080:1920:"
                "force_original_aspect_ratio=increase,"
                "crop=1080:1920,"
                + zoom_filter
            ),
            "-t", str(image_duration),
            "-an",
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-r", "30",
            output
        ]

        result = subprocess.run(
            command,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE
        )

        if result.returncode != 0:
            print(result.stderr.decode(
                "utf-8",
                errors="ignore"
            ))
            raise RuntimeError(
                f"Failed to create clip for {image}"
            )

        generated_clips.append(output)

        print(
            f"✓ Image {index + 1}/{len(image_files)}"
        )

    print("\n========================================")
    print("✅ IMAGE MOTION READY")
    print("========================================")

    print(
        f"Generated clips: {len(generated_clips)}"
    )

🎬 CREATING IMAGE MOTION
Images: 8
Time per image: 1.04s
✓ Image 1/8
✓ Image 2/8
✓ Image 3/8
✓ Image 4/8
✓ Image 5/8
✓ Image 6/8
✓ Image 7/8
✓ Image 8/8

✅ IMAGE MOTION READY
Generated clips: 8


In [ ]:

# ============================================================
# CELL 10 — VIDEO INPUT PROCESSING
# ============================================================

import os
import subprocess

VIDEO_WIDTH = 1080
VIDEO_HEIGHT = 1920

VIDEO_CLIP = "/content/sc_pipeline/temp/input_video.mp4"

if not video_files:

    print("ℹ️ No video input found.")
    print("Images will be used instead.")

else:

    input_video = video_files[0]

    print("========================================")
    print("🎥 PROCESSING VIDEO INPUT")
    print("========================================")

    print("Input:", os.path.basename(input_video))

    command = [
        "ffmpeg",
        "-y",
        "-i", input_video,

        # Fill the 9:16 frame without distortion
        "-vf",
        (
            "scale=1080:1920:"
            "force_original_aspect_ratio=increase,"
            "crop=1080:1920"
        ),

        # Remove original audio
        "-an",

        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "20",
        "-pix_fmt", "yuv420p",
        "-r", "30",

        VIDEO_CLIP
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE
    )

    if result.returncode != 0:

        print(
            result.stderr.decode(
                "utf-8",
                errors="ignore"
            )
        )

        raise RuntimeError(
            "Video processing failed."
        )

    if not os.path.exists(VIDEO_CLIP):
        raise RuntimeError(
            "Processed video was not created."
        )

    print("\n========================================")
    print("✅ VIDEO READY")
    print("========================================")

    print("Output:", VIDEO_CLIP)
    print("Resolution: 1080 × 1920")
    print("Original audio: MUTED")

ℹ️ No video input found.
Images will be used instead.


In [ ]:

# ============================================================
# CELL 11 — PROFESSIONAL MULTI-TRANSITIONS
# ============================================================

import os
import subprocess

IMAGE_CLIPS_DIR = "/content/sc_pipeline/temp/image_clips"
TRANSITION_DIR = "/content/sc_pipeline/temp/transitions"

os.makedirs(TRANSITION_DIR, exist_ok=True)

# ------------------------------------------------------------
# FIND ANIMATED IMAGE CLIPS
# ------------------------------------------------------------

image_clips = sorted(
    [
        os.path.join(IMAGE_CLIPS_DIR, f)
        for f in os.listdir(IMAGE_CLIPS_DIR)
        if f.lower().endswith(".mp4")
    ]
)

if not image_clips:
    print("ℹ️ No image clips found.")
    print("Video input will be handled separately.")

else:

    if len(image_clips) == 1:

        FINAL_VISUAL = image_clips[0]

        print("ℹ️ Only one image.")
        print("No transition required.")

    else:

        # ----------------------------------------------------
        # TRANSITION SETTINGS
        # ----------------------------------------------------

        TRANSITION_DURATION = 0.45

        # Professional transitions suitable for sports news
        TRANSITIONS = [
            "fade",
            "slideleft",
            "slideright",
            "wipeleft",
            "wiperight",
            "zoomin"
        ]

        current_clip = image_clips[0]

        print("========================================")
        print("✨ PROFESSIONAL TRANSITIONS")
        print("========================================")

        print(f"Images: {len(image_clips)}")
        print(
            f"Transition duration: "
            f"{TRANSITION_DURATION}s"
        )

        print("\nTransition sequence:")

        # ----------------------------------------------------
        # CREATE TRANSITIONS
        # ----------------------------------------------------

        for index in range(1, len(image_clips)):

            next_clip = image_clips[index]

            transition_name = TRANSITIONS[
                (index - 1) % len(TRANSITIONS)
            ]

            output = os.path.join(
                TRANSITION_DIR,
                f"transition_{index:03d}.mp4"
            )

            # -----------------------------------------------
            # GET CURRENT CLIP DURATION
            # -----------------------------------------------

            probe = subprocess.run(
                [
                    "ffprobe",
                    "-v", "error",
                    "-show_entries",
                    "format=duration",
                    "-of",
                    "default=noprint_wrappers=1:nokey=1",
                    current_clip
                ],
                capture_output=True,
                text=True
            )

            if probe.returncode != 0:
                raise RuntimeError(
                    "Could not determine clip duration."
                )

            current_duration = float(
                probe.stdout.strip()
            )

            offset = max(
                0,
                current_duration - TRANSITION_DURATION
            )

            # -----------------------------------------------
            # BUILD XFADE
            # -----------------------------------------------

            filter_complex = (
                f"[0:v][1:v]"
                f"xfade="
                f"transition={transition_name}:"
                f"duration={TRANSITION_DURATION}:"
                f"offset={offset},"
                f"format=yuv420p[v]"
            )

            command = [
                "ffmpeg",
                "-y",

                "-i", current_clip,
                "-i", next_clip,

                "-filter_complex",
                filter_complex,

                "-map", "[v]",
                "-an",

                "-c:v", "libx264",
                "-preset", "veryfast",
                "-crf", "20",
                "-pix_fmt", "yuv420p",
                "-r", "30",

                output
            ]

            result = subprocess.run(
                command,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.PIPE
            )

            if result.returncode != 0:

                print(
                    result.stderr.decode(
                        "utf-8",
                        errors="ignore"
                    )
                )

                raise RuntimeError(
                    f"Transition failed: "
                    f"{transition_name}"
                )

            current_clip = output

            print(
                f"✓ {index}: "
                f"{transition_name}"
            )

        FINAL_VISUAL = current_clip

    print("\n========================================")
    print("✅ MULTI-TRANSITION VISUAL READY")
    print("========================================")

    print("Final visual:", FINAL_VISUAL)

✨ PROFESSIONAL TRANSITIONS
Images: 8
Transition duration: 0.45s

Transition sequence:
✓ 1: fade
✓ 2: slideleft
✓ 3: slideright
✓ 4: wipeleft
✓ 5: wiperight
✓ 6: zoomin
✓ 7: fade

✅ MULTI-TRANSITION VISUAL READY
Final visual: /content/sc_pipeline/temp/transitions/transition_007.mp4


In [ ]:

# ============================================================
# CELL 12 — DETECT MEDIA TYPE & SELECT VISUAL TRACK
# ============================================================

import os

print("========================================")
print("🔍 DETECTING MEDIA TYPE")
print("========================================")

if video_files:

    # One uploaded video
    MEDIA_TYPE = "video"
    FINAL_VISUAL = VIDEO_CLIP

    print("🎥 Media type: VIDEO")
    print("Using:", os.path.basename(VIDEO_CLIP))

elif image_files:

    # Uploaded images
    MEDIA_TYPE = "images"

    if "FINAL_VISUAL" not in globals():
        raise RuntimeError(
            "Image visual track was not created. "
            "Check Cell 11."
        )

    print("📸 Media type: IMAGES")
    print(f"Images detected: {len(image_files)}")
    print("Using animated image sequence.")

else:

    raise RuntimeError(
        "No supported media was detected."
    )

# ------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------

if not os.path.exists(FINAL_VISUAL):
    raise FileNotFoundError(
        f"Final visual track not found:\n{FINAL_VISUAL}"
    )

print("\n========================================")
print("✅ MEDIA TYPE DETECTED")
print("========================================")
print("Type:", MEDIA_TYPE)
print("Visual:", FINAL_VISUAL)

🔍 DETECTING MEDIA TYPE
📸 Media type: IMAGES
Images detected: 8
Using animated image sequence.

✅ MEDIA TYPE DETECTED
Type: images
Visual: /content/sc_pipeline/temp/transitions/transition_007.mp4


In [ ]:

# ============================================================
# CELL 13 — HOOK / TITLE CARD
# ============================================================

import os
import subprocess

HOOK_DIR = "/content/sc_pipeline/temp/hook"
os.makedirs(HOOK_DIR, exist_ok=True)

HOOK_VIDEO = os.path.join(
    HOOK_DIR,
    "hook.mp4"
)

if not USE_HOOK:

    HOOK_DURATION_ACTUAL = 0
    HOOK_VIDEO = None

    print("ℹ️ Hook disabled.")

else:

    print("========================================")
    print("🔥 CREATING VIDEO HOOK")
    print("========================================")

    # Escape text safely for FFmpeg
    hook_text = HOOK_TEXT.replace(":", r"\:")
    hook_subtext = HOOK_SUBTEXT.replace(":", r"\:")

    filter_complex = (
        "[0:v]"
        "format=yuv420p,"
        "drawtext="
        "fontfile=/usr/share/fonts/truetype/dejavu/"
        "DejaVuSans-Bold.ttf:"
        f"text='{hook_text}':"
        "fontcolor=white:"
        "fontsize=72:"
        "x=(w-text_w)/2:"
        "y=(h-text_h)/2-70:"
        "box=1:"
        "boxcolor=black@0.55:"
        "boxborderw=22,"
        "drawtext="
        "fontfile=/usr/share/fonts/truetype/dejavu/"
        "DejaVuSans-Bold.ttf:"
        f"text='{hook_subtext}':"
        "fontcolor=white:"
        "fontsize=42:"
        "x=(w-text_w)/2:"
        "y=(h-text_h)/2+35:"
        "box=1:"
        "boxcolor=black@0.55:"
        "boxborderw=16"
    )

    command = [
        "ffmpeg",
        "-y",

        "-f", "lavfi",
        "-i",
        f"color=c=black:s={VIDEO_WIDTH}x{VIDEO_HEIGHT}:"
        f"d={HOOK_DURATION}:r=30",

        "-vf",
        filter_complex,

        "-an",

        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "20",
        "-pix_fmt", "yuv420p",
        "-r", "30",

        HOOK_VIDEO
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE
    )

    if result.returncode != 0:

        print(
            result.stderr.decode(
                "utf-8",
                errors="ignore"
            )
        )

        raise RuntimeError(
            "Hook creation failed."
        )

    HOOK_DURATION_ACTUAL = HOOK_DURATION

    print("✓ Hook created")
    print("Duration:", HOOK_DURATION_ACTUAL, "seconds")
    print("Main text:", HOOK_TEXT)
    print("Subtext:", HOOK_SUBTEXT)

print("\n========================================")
print("✅ HOOK READY")
print("========================================")

🔥 CREATING VIDEO HOOK
✓ Hook created
Duration: 1.5 seconds
Main text: BROOKE QUARLES DANIELS
Subtext: IS PUTTING IN THE WORK

✅ HOOK READY


In [ ]:

# ============================================================
# CELL 14 — BACKGROUND MUSIC
# ============================================================

import os
import shutil

PERSISTENT_MUSIC_DIR = (
    "/content/drive/MyDrive/"
    "Sc_Gamecocks_Video_Pipeline/music"
)

MUSIC_DIR = "/content/sc_pipeline/music"

os.makedirs(PERSISTENT_MUSIC_DIR, exist_ok=True)
os.makedirs(MUSIC_DIR, exist_ok=True)

MUSIC_FILE = None

if not ADD_MUSIC:

    print("ℹ️ Background music disabled.")

else:

    music_extensions = (
        ".mp3",
        ".wav",
        ".m4a",
        ".aac",
        ".ogg"
    )

    music_files = sorted(
        [
            os.path.join(PERSISTENT_MUSIC_DIR, f)
            for f in os.listdir(PERSISTENT_MUSIC_DIR)
            if f.lower().endswith(music_extensions)
        ]
    )

    if not music_files:

        print("⚠️ No music found in Drive.")
        print(
            "Upload a music file to:\n"
            f"{PERSISTENT_MUSIC_DIR}"
        )

        ADD_MUSIC = False

    else:

        # Use the first available music track
        source_music = music_files[0]

        MUSIC_FILE = os.path.join(
            MUSIC_DIR,
            os.path.basename(source_music)
        )

        shutil.copy2(
            source_music,
            MUSIC_FILE
        )

        print("========================================")
        print("🎵 BACKGROUND MUSIC")
        print("========================================")

        print(
            "Selected:",
            os.path.basename(source_music)
        )

        print(
            "Volume:",
            MUSIC_VOLUME
        )

        print("\n✓ Music ready")

print("\n========================================")
print("✅ MUSIC SETUP COMPLETE")
print("========================================")

⚠️ No music found in Drive.
Upload a music file to:
/content/drive/MyDrive/Sc_Gamecocks_Video_Pipeline/music

✅ MUSIC SETUP COMPLETE


In [ ]:

# ============================================================
# CELL 15 — NARRATION + MUSIC MIX
# ============================================================

import os
import subprocess

FINAL_AUDIO = "/content/sc_pipeline/temp/final_audio.wav"

NARRATION_AUDIO = "/content/narration.wav"

if not os.path.exists(NARRATION_AUDIO):
    raise FileNotFoundError(
        "Narration audio not found. "
        "Run Cell 6 first."
    )

print("========================================")
print("🎚️ AUDIO MIX")
print("========================================")

# ------------------------------------------------------------
# NARRATION ONLY
# ------------------------------------------------------------

if not ADD_MUSIC or MUSIC_FILE is None:

    print("🎙️ Narration only")

    command = [
        "ffmpeg",
        "-y",
        "-i", NARRATION_AUDIO,
        "-af",
        (
            "highpass=f=70,"
            "lowpass=f=14000,"
            "acompressor="
            "threshold=0.08:"
            "ratio=2.5:"
            "attack=15:"
            "release=180,"
            "loudnorm=I=-16:LRA=7:TP=-1.5"
        ),
        "-ar", "48000",
        "-ac", "2",
        "-c:a", "pcm_s16le",
        FINAL_AUDIO
    ]

# ------------------------------------------------------------
# NARRATION + DUCKED MUSIC
# ------------------------------------------------------------

else:

    print("🎙️ Narration + 🎵 background music")
    print("Music ducking enabled.")

    command = [
        "ffmpeg",
        "-y",

        "-i", NARRATION_AUDIO,
        "-stream_loop", "-1",
        "-i", MUSIC_FILE,

        "-filter_complex",

        (
            # Narration processing
            "[0:a]"
            "highpass=f=70,"
            "lowpass=f=14000,"
            "acompressor="
            "threshold=0.08:"
            "ratio=2.5:"
            "attack=15:"
            "release=180,"
            "loudnorm=I=-16:LRA=7:TP=-1.5"
            "[voice];"

            # Music processing
            f"[1:a]"
            f"volume={MUSIC_VOLUME},"
            "highpass=f=40,"
            "lowpass=f=12000"
            "[music];"

            # Automatic ducking
            "[music][voice]"
            "sidechaincompress="
            "threshold=0.02:"
            "ratio=8:"
            "attack=20:"
            "release=350:"
            "makeup=1"
            "[ducked];"

            # Final mix
            "[voice][ducked]"
            "amix=inputs=2:"
            "duration=first:"
            "dropout_transition=2,"
            "loudnorm=I=-16:LRA=7:TP=-1.5"
            "[out]"
        ),

        "-map", "[out]",

        "-ar", "48000",
        "-ac", "2",
        "-c:a", "pcm_s16le",

        FINAL_AUDIO
    ]

result = subprocess.run(
    command,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

if result.returncode != 0:

    print(
        result.stderr.decode(
            "utf-8",
            errors="ignore"
        )
    )

    raise RuntimeError(
        "Audio mixing failed."
    )

if not os.path.exists(FINAL_AUDIO):
    raise RuntimeError(
        "Final audio was not created."
    )

print("\n========================================")
print("✅ AUDIO MIX READY")
print("========================================")
print("Output:", FINAL_AUDIO)

🎚️ AUDIO MIX
🎙️ Narration only

✅ AUDIO MIX READY
Output: /content/sc_pipeline/temp/final_audio.wav


In [ ]:

# ============================================================
# CELL 16 — WHISPER WORD TIMESTAMPS
# ============================================================

import os
import whisper

NARRATION_AUDIO = "/content/narration.wav"

if not os.path.exists(NARRATION_AUDIO):
    raise FileNotFoundError(
        "Narration audio not found. Run Cell 6 first."
    )

print("========================================")
print("🎙️ WHISPER TRANSCRIPTION")
print("========================================")

# Use the persistent model prepared by Cell 3
if "whisper_model" not in globals():

    print("Loading Whisper base model...")

    whisper_model = whisper.load_model(
        "base"
    )

print("Transcribing narration...")
print("Word timestamps enabled.")

TRANSCRIPTION = whisper_model.transcribe(
    NARRATION_AUDIO,
    language="en",
    word_timestamps=True,
    fp16=False
)

WORD_SEGMENTS = []

for segment in TRANSCRIPTION.get("segments", []):

    for word in segment.get("words", []):

        text = word.get("word", "").strip()

        if not text:
            continue

        WORD_SEGMENTS.append({
            "word": text,
            "start": float(word["start"]),
            "end": float(word["end"])
        })

if not WORD_SEGMENTS:
    raise RuntimeError(
        "Whisper did not return word timestamps."
    )

print("\n========================================")
print("✅ WHISPER READY")
print("========================================")

print("Words detected:", len(WORD_SEGMENTS))

print("\nFirst words:")

for item in WORD_SEGMENTS[:10]:
    print(
        f"{item['start']:.2f}s → "
        f"{item['end']:.2f}s  "
        f"{item['word']}"
    )

🎙️ WHISPER TRANSCRIPTION
Transcribing narration...
Word timestamps enabled.

✅ WHISPER READY
Words detected: 25

First words:
0.00s → 0.20s  Brooke
0.20s → 0.72s  Quarrell's
0.72s → 1.22s  Daniels
1.22s → 1.28s  is
1.28s → 1.64s  already
1.64s → 1.98s  putting
1.98s → 2.18s  in
2.18s → 2.24s  the
2.24s → 2.50s  work.
3.06s → 3.32s  The


In [ ]:

# ============================================================
# CELL 17 — ANIMATED KARAOKE CAPTIONS
# ============================================================

import os
import re

ASS_FILE = "/content/sc_pipeline/temp/captions.ass"

os.makedirs(
    "/content/sc_pipeline/temp",
    exist_ok=True
)

if not WORD_SEGMENTS:
    raise RuntimeError(
        "No Whisper word timestamps found. "
        "Check Cell 16."
    )

# ------------------------------------------------------------
# CAPTION SETTINGS
# ------------------------------------------------------------

FONT_NAME = "DejaVu Sans"
FONT_SIZE = 58

SCREEN_WIDTH = 1080
SCREEN_HEIGHT = 1920

# Caption position
CAPTION_X = 540
CAPTION_Y = 1580

# Maximum words displayed at once
WORDS_PER_LINE = 5

# ------------------------------------------------------------
# ASS HEADER
# ------------------------------------------------------------

ass_header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: {SCREEN_WIDTH}
PlayResY: {SCREEN_HEIGHT}
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,{FONT_NAME},{FONT_SIZE},&H00FFFFFF,&H00FFFFFF,&H80000000,&H80000000,1,0,0,0,100,100,0,0,1,4,1,5,40,40,40,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def ass_time(seconds):
    seconds = max(0, float(seconds))

    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    centiseconds = int((seconds - int(seconds)) * 100)

    return (
        f"{hours}:{minutes:02d}:"
        f"{secs:02d}.{centiseconds:02d}"
    )


def clean_word(word):
    return (
        word.replace("{", "")
            .replace("}", "")
            .replace("\\", "")
    )


# ------------------------------------------------------------
# BUILD CAPTION EVENTS
# ------------------------------------------------------------

events = []

for start_index in range(
    0,
    len(WORD_SEGMENTS),
    WORDS_PER_LINE
):

    group = WORD_SEGMENTS[
        start_index:start_index + WORDS_PER_LINE
    ]

    if not group:
        continue

    group_start = group[0]["start"]
    group_end = group[-1]["end"]

    # Create one caption event for each word
    # so the active word can be highlighted.
    for active_index, active_word in enumerate(group):

        active_start = active_word["start"]

        if active_index + 1 < len(group):
            active_end = group[
                active_index + 1
            ]["start"]
        else:
            active_end = group_end

        words = []

        for index, item in enumerate(group):

            word = clean_word(item["word"])

            if index == active_index:
                # Highlight the currently spoken word
                words.append(
                    r"{\c&H00FFFF&}" +
                    word +
                    r"{\c&HFFFFFF&}"
                )
            else:
                words.append(word)

        caption_text = " ".join(words)

        events.append(
            "Dialogue: 0,"
            f"{ass_time(active_start)},"
            f"{ass_time(active_end)},"
            f"Default,,0,0,0,,"
            f"{{\\pos({CAPTION_X},{CAPTION_Y})}}"
            f"{caption_text}"
        )

# ------------------------------------------------------------
# WRITE ASS FILE
# ------------------------------------------------------------

with open(
    ASS_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(ass_header)
    f.write("\n".join(events))

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

if not os.path.exists(ASS_FILE):
    raise RuntimeError(
        "Caption ASS file was not created."
    )

print("========================================")
print("💬 KARAOKE CAPTIONS READY")
print("========================================")

print("Words:", len(WORD_SEGMENTS))
print("Caption events:", len(events))
print("Font:", FONT_NAME)
print("Font size:", FONT_SIZE)
print("Position:", f"{CAPTION_X}, {CAPTION_Y}")
print("ASS file:", ASS_FILE)

print("\n✓ Word-by-word highlighting enabled")

💬 KARAOKE CAPTIONS READY
Words: 25
Caption events: 25
Font: DejaVu Sans
Font size: 58
Position: 540, 1580
ASS file: /content/sc_pipeline/temp/captions.ass

✓ Word-by-word highlighting enabled


In [ ]:

# ============================================================
# CELL 18 — SYNC CAPTIONS WITH HOOK
# ============================================================

import os
import re

SOURCE_ASS = "/content/sc_pipeline/temp/captions.ass"
SYNCED_ASS = "/content/sc_pipeline/temp/captions_synced.ass"

if not os.path.exists(SOURCE_ASS):
    raise FileNotFoundError(
        "Caption file not found. Check Cell 17."
    )

# If there is no hook, no timing shift is needed
CAPTION_OFFSET = (
    HOOK_DURATION_ACTUAL
    if USE_HOOK
    else 0
)

def shift_ass_time(time_string, offset):
    """
    Shift an ASS timestamp by offset seconds.
    ASS format: H:MM:SS.cc
    """

    hours, minutes, seconds = time_string.split(":")

    seconds = float(seconds)

    total_seconds = (
        int(hours) * 3600
        + int(minutes) * 60
        + seconds
        + offset
    )

    total_seconds = max(0, total_seconds)

    new_hours = int(total_seconds // 3600)

    new_minutes = int(
        (total_seconds % 3600) // 60
    )

    new_seconds = (
        total_seconds
        - new_hours * 3600
        - new_minutes * 60
    )

    return (
        f"{new_hours}:"
        f"{new_minutes:02d}:"
        f"{new_seconds:05.2f}"
    )


with open(
    SOURCE_ASS,
    "r",
    encoding="utf-8"
) as f:

    lines = f.readlines()


shifted_lines = []

for line in lines:

    if not line.startswith("Dialogue:"):
        shifted_lines.append(line)
        continue

    parts = line.rstrip("\n").split(",", 9)

    if len(parts) < 10:
        shifted_lines.append(line)
        continue

    start_time = parts[1]
    end_time = parts[2]

    parts[1] = shift_ass_time(
        start_time,
        CAPTION_OFFSET
    )

    parts[2] = shift_ass_time(
        end_time,
        CAPTION_OFFSET
    )

    shifted_lines.append(
        ",".join(parts) + "\n"
    )


with open(
    SYNCED_ASS,
    "w",
    encoding="utf-8"
) as f:

    f.writelines(shifted_lines)


if not os.path.exists(SYNCED_ASS):
    raise RuntimeError(
        "Synced caption file was not created."
    )

print("========================================")
print("⏱️ CAPTIONS SYNCHRONIZED")
print("========================================")

print(
    "Hook offset:",
    round(CAPTION_OFFSET, 2),
    "seconds"
)

print("Caption file:", SYNCED_ASS)

print("\n✓ Caption timing adjusted")

⏱️ CAPTIONS SYNCHRONIZED
Hook offset: 1.5 seconds
Caption file: /content/sc_pipeline/temp/captions_synced.ass

✓ Caption timing adjusted


In [ ]:

# ============================================================
# CELL 19 — FINAL VIDEO RENDER
# ============================================================

import os
import subprocess

OUTPUT = (
    "/content/drive/MyDrive/"
    "Sc_Gamecocks_Video_Pipeline/outputs/"
    "gamecocks_story.mp4"
)

os.makedirs(
    os.path.dirname(OUTPUT),
    exist_ok=True
)

if not os.path.exists(FINAL_VISUAL):
    raise FileNotFoundError(
        f"Visual track not found:\n{FINAL_VISUAL}"
    )

if not os.path.exists(FINAL_AUDIO):
    raise FileNotFoundError(
        f"Final audio not found:\n{FINAL_AUDIO}"
    )

if USE_HOOK and not os.path.exists(HOOK_VIDEO):
    raise FileNotFoundError(
        f"Hook video not found:\n{HOOK_VIDEO}"
    )

print("========================================")
print("🎬 FINAL VIDEO RENDER")
print("========================================")

# ------------------------------------------------------------
# STEP 1 — COMBINE HOOK + MAIN VISUAL
# ------------------------------------------------------------

VISUAL_WITH_HOOK = (
    "/content/sc_pipeline/temp/"
    "visual_with_hook.mp4"
)

if USE_HOOK:

    print("🔥 Adding opening hook...")

    concat_file = (
        "/content/sc_pipeline/temp/"
        "visual_concat.txt"
    )

    with open(
        concat_file,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            f"file '{HOOK_VIDEO}'\n"
        )

        f.write(
            f"file '{FINAL_VISUAL}'\n"
        )

    command = [
        "ffmpeg",
        "-y",

        "-f", "concat",
        "-safe", "0",
        "-i", concat_file,

        "-an",

        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "20",
        "-pix_fmt", "yuv420p",
        "-r", "30",

        VISUAL_WITH_HOOK
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE
    )

    if result.returncode != 0:

        print(
            result.stderr.decode(
                "utf-8",
                errors="ignore"
            )
        )

        raise RuntimeError(
            "Hook + visual combination failed."
        )

else:

    VISUAL_WITH_HOOK = FINAL_VISUAL

# ------------------------------------------------------------
# STEP 2 — FINAL VIDEO + AUDIO + CAPTIONS
# ------------------------------------------------------------

print("🎙️ Adding final audio...")
print("💬 Burning synchronized captions...")

CAPTION_FILE = (
    "/content/sc_pipeline/temp/"
    "captions_synced.ass"
)

if not os.path.exists(CAPTION_FILE):
    raise FileNotFoundError(
        "Synchronized captions not found."
    )

# Escape caption path for FFmpeg filter syntax
caption_filter_path = (
    CAPTION_FILE
    .replace("\\", "/")
    .replace(":", "\\:")
)

command = [
    "ffmpeg",
    "-y",

    "-i", VISUAL_WITH_HOOK,
    "-i", FINAL_AUDIO,

    "-vf",
    f"ass='{caption_filter_path}'",

    "-map", "0:v",
    "-map", "1:a",

    "-c:v", "libx264",
    "-preset", "veryfast",
    "-crf", "20",

    "-c:a", "aac",
    "-b:a", "192k",
    "-ar", "48000",

    "-pix_fmt", "yuv420p",

    "-shortest",

    OUTPUT
]

result = subprocess.run(
    command,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

if result.returncode != 0:

    print(
        result.stderr.decode(
            "utf-8",
            errors="ignore"
        )
    )

    raise RuntimeError(
        "Final video render failed."
    )

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

if not os.path.exists(OUTPUT):
    raise RuntimeError(
        "Final video was not created."
    )

file_size_mb = (
    os.path.getsize(OUTPUT)
    / (1024 * 1024)
)

print("\n========================================")
print("✅ FINAL VIDEO READY")
print("========================================")

print("Output:", OUTPUT)
print(
    f"File size: {file_size_mb:.2f} MB"
)
print("Resolution: 1080 × 1920")
print("Format: 9:16")
print("Audio: Narration + Music")
print("Captions: Animated")
print("Hook:", USE_HOOK)

print("\n🎉 RENDER COMPLETE")

🎬 FINAL VIDEO RENDER
🔥 Adding opening hook...
🎙️ Adding final audio...
💬 Burning synchronized captions...

✅ FINAL VIDEO READY
Output: /content/drive/MyDrive/Sc_Gamecocks_Video_Pipeline/outputs/gamecocks_story.mp4
File size: 1.52 MB
Resolution: 1080 × 1920
Format: 9:16
Audio: Narration + Music
Captions: Animated
Hook: True

🎉 RENDER COMPLETE


In [ ]:

# ============================================================
# CELL 20 — FINAL VIDEO PREVIEW
# ============================================================

import os
from IPython.display import Video, display

OUTPUT = (
    "/content/drive/MyDrive/"
    "Sc_Gamecocks_Video_Pipeline/outputs/"
    "gamecocks_story.mp4"
)

if not os.path.exists(OUTPUT):
    raise FileNotFoundError(
        "Final video not found.\n"
        "Run Cell 19 first."
    )

file_size_mb = (
    os.path.getsize(OUTPUT)
    / (1024 * 1024)
)

print("========================================")
print("🎬 FINAL VIDEO")
print("========================================")

print("File:", OUTPUT)
print(f"Size: {file_size_mb:.2f} MB")

print("\n▶️ Preview:")

display(
    Video(
        OUTPUT,
        embed=True,
        width=360
    )
)

print("\n========================================")
print("✅ VIDEO READY")
print("========================================")

🎬 FINAL VIDEO
File: /content/drive/MyDrive/Sc_Gamecocks_Video_Pipeline/outputs/gamecocks_story.mp4
Size: 1.52 MB

▶️ Preview:



✅ VIDEO READY


In [4]:

# ============================================================
# CELL 21 — GRADIO VIDEO MAKER — UPGRADED
# ============================================================

import os
import re
import shutil
import subprocess
import gradio as gr

# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "Sc_Gamecocks_Video_Pipeline"
)

PERSISTENT_MEDIA_DIR = os.path.join(
    PROJECT_DIR, "media"
)

PERSISTENT_MUSIC_DIR = os.path.join(
    PROJECT_DIR, "music"
)

PERSISTENT_OUTPUT_DIR = os.path.join(
    PROJECT_DIR, "outputs"
)

RUNTIME_DIR = "/content/sc_pipeline"

RUNTIME_MEDIA_DIR = os.path.join(
    RUNTIME_DIR, "media"
)

RUNTIME_TEMP_DIR = os.path.join(
    RUNTIME_DIR, "temp"
)

os.makedirs(PERSISTENT_MEDIA_DIR, exist_ok=True)
os.makedirs(PERSISTENT_MUSIC_DIR, exist_ok=True)
os.makedirs(PERSISTENT_OUTPUT_DIR, exist_ok=True)
os.makedirs(RUNTIME_MEDIA_DIR, exist_ok=True)
os.makedirs(RUNTIME_TEMP_DIR, exist_ok=True)


# ============================================================
# PIPER VOICES
# ============================================================

def find_piper_voices():

    voices = {}

    if os.path.exists("/content/voices"):

        for filename in os.listdir("/content/voices"):

            if filename.endswith(".onnx"):

                name = filename[:-5]

                voices[name] = os.path.join(
                    "/content/voices",
                    filename
                )

    # Preserve current working voice
    default_voice = (
        "/content/voices/"
        "en_US-lessac-medium.onnx"
    )

    if os.path.exists(default_voice):

        voices.setdefault(
            "en_US-lessac-medium",
            default_voice
        )

    return voices


PIPER_VOICES = find_piper_voices()

if not PIPER_VOICES:

    raise RuntimeError(
        "No Piper voices found. Run Cell 2 first."
    )


# ============================================================
# COMMAND HELPER
# ============================================================

def run_cmd(command, input_text=None):

    result = subprocess.run(
        command,
        input=input_text,
        text=True if input_text is not None else False,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    if result.returncode != 0:

        error = result.stderr

        if isinstance(error, bytes):
            error = error.decode(
                "utf-8",
                errors="ignore"
            )

        raise RuntimeError(error)

    return result


# ============================================================
# DURATION
# ============================================================

def get_duration(path):

    result = run_cmd([
        "ffprobe",
        "-v", "error",
        "-show_entries",
        "format=duration",
        "-of",
        "default=noprint_wrappers=1:nokey=1",
        path
    ])

    return float(
        result.stdout.strip()
    )


# ============================================================
# ASS TIME
# ============================================================

def ass_time(seconds):

    seconds = max(0, float(seconds))

    hours = int(seconds // 3600)

    minutes = int(
        (seconds % 3600) // 60
    )

    secs = (
        seconds
        - hours * 3600
        - minutes * 60
    )

    return (
        f"{hours}:"
        f"{minutes:02d}:"
        f"{secs:05.2f}"
    )


# ============================================================
# PACING PRESETS
# ============================================================

PACING_SETTINGS = {

    "Breaking News": {
        "sentence_pause": 0.08,
        "paragraph_pause": 0.20
    },

    "Normal Sports Story": {
        "sentence_pause": 0.18,
        "paragraph_pause": 0.40
    },

    "High Energy": {
        "sentence_pause": 0.04,
        "paragraph_pause": 0.12
    },

    "Analysis": {
        "sentence_pause": 0.28,
        "paragraph_pause": 0.55
    }
}


# ============================================================
# NARRATION
# ============================================================

def create_narration(
    script,
    speech_rate,
    voice_name,
    pacing
):

    if not script.strip():

        raise RuntimeError(
            "Please enter a script."
        )

    speech_rate = float(
        speech_rate
    )

    if not 0.85 <= speech_rate <= 1.15:

        raise RuntimeError(
            "Narration speed must be between "
            "0.85x and 1.15x."
        )

    voice = PIPER_VOICES.get(
        voice_name
    )

    if not voice:

        raise RuntimeError(
            f"Piper voice not found: {voice_name}"
        )

    settings = PACING_SETTINGS.get(
        pacing,
        PACING_SETTINGS[
            "Normal Sports Story"
        ]
    )

    sentence_pause = settings[
        "sentence_pause"
    ]

    temp_dir = RUNTIME_TEMP_DIR

    for filename in os.listdir(temp_dir):

        if (
            filename.startswith("sentence_")
            and filename.endswith(".wav")
        ):

            os.remove(
                os.path.join(
                    temp_dir,
                    filename
                )
            )

    text = script.replace(
        "\r\n",
        "\n"
    ).replace(
        "\r",
        "\n"
    )

    paragraphs = [
        p.strip()
        for p in re.split(
            r"\n\s*\n",
            text
        )
        if p.strip()
    ]

    sentences = []

    for paragraph in paragraphs:

        parts = re.split(
            r"(?<=[.!?])\s+",
            paragraph
        )

        for part in parts:

            part = part.strip()

            if part:
                sentences.append(part)

    if not sentences:

        raise RuntimeError(
            "No usable sentences found."
        )

    sentence_files = []

    print("\n🎙️ CREATING NARRATION")
    print("Voice:", voice_name)
    print(f"Speed: {speech_rate:.2f}x")
    print("Pacing:", pacing)

    for index, sentence in enumerate(
        sentences
    ):

        output = os.path.join(
            temp_dir,
            f"sentence_{index:03d}.wav"
        )

        print(
            f"  {index + 1}/"
            f"{len(sentences)}: "
            f"{sentence}"
        )

        run_cmd(
            [
                "piper",
                "--model",
                voice,
                "--length-scale",
                str(1.0 / speech_rate),
                "--output_file",
                output
            ],
            input_text=sentence
        )

        sentence_files.append(
            output
        )

    silence = os.path.join(
        temp_dir,
        "sentence_silence.wav"
    )

    run_cmd([
        "ffmpeg",
        "-y",
        "-f",
        "lavfi",
        "-i",
        "anullsrc=r=48000:cl=mono",
        "-t",
        str(sentence_pause),
        "-c:a",
        "pcm_s16le",
        silence
    ])

    concat_file = os.path.join(
        temp_dir,
        "narration_concat.txt"
    )

    with open(
        concat_file,
        "w",
        encoding="utf-8"
    ) as f:

        for index, filename in enumerate(
            sentence_files
        ):

            f.write(
                f"file '{filename}'\n"
            )

            if index < len(sentence_files) - 1:

                f.write(
                    f"file '{silence}'\n"
                )

    raw_narration = os.path.join(
        temp_dir,
        "narration_raw.wav"
    )

    run_cmd([
        "ffmpeg",
        "-y",
        "-f",
        "concat",
        "-safe",
        "0",
        "-i",
        concat_file,
        "-ar",
        "48000",
        "-ac",
        "1",
        "-c:a",
        "pcm_s16le",
        raw_narration
    ])

    narration = os.path.join(
        RUNTIME_DIR,
        "narration.wav"
    )

    run_cmd([
        "ffmpeg",
        "-y",
        "-i",
        raw_narration,
        "-af",
        (
            "highpass=f=70,"
            "lowpass=f=14000,"
            "acompressor="
            "threshold=0.08:"
            "ratio=2.5:"
            "attack=15:"
            "release=180:"
            "makeup=1,"
            "loudnorm="
            "I=-16:"
            "LRA=7:"
            "TP=-1.5"
        ),
        "-ar",
        "48000",
        "-ac",
        "1",
        "-c:a",
        "pcm_s16le",
        narration
    ])

    print(
        "✓ Narration ready:",
        round(
            get_duration(narration),
            2
        ),
        "seconds"
    )

    return narration


# ============================================================
# CAPTIONS
# ============================================================

def create_captions(
    narration,
    font_size,
    words_per_line,
    caption_position,
    caption_style
):

    print("\n💬 CREATING CAPTIONS")

    import whisper

    if "whisper_model" in globals():

        model = whisper_model

    else:

        print("Loading Whisper...")

        model = whisper.load_model(
            "base"
        )

    result = model.transcribe(
        narration,
        language="en",
        word_timestamps=True,
        fp16=False
    )

    words = []

    for segment in result.get(
        "segments",
        []
    ):

        for word in segment.get(
            "words",
            []
        ):

            text = word.get(
                "word",
                ""
            ).strip()

            if text:

                words.append({
                    "word": text,
                    "start": float(
                        word["start"]
                    ),
                    "end": float(
                        word["end"]
                    )
                })

    if not words:

        raise RuntimeError(
            "Whisper returned no word timestamps."
        )

    positions = {
        "Top": 300,
        "Center": 960,
        "Bottom": 1580
    }

    y_position = positions.get(
        caption_position,
        1580
    )

    styles = {

        "Karaoke": (
            "&H00FFFFFF",
            "&H00FFFF00",
            "1"
        ),

        "Clean": (
            "&H00FFFFFF",
            "&H00FFFFFF",
            "1"
        ),

        "Bold": (
            "&H00FFFFFF",
            "&H0000FFFF",
            "1"
        )
    }

    primary, secondary, bold = styles.get(
        caption_style,
        styles["Karaoke"]
    )

    ass_file = os.path.join(
        RUNTIME_TEMP_DIR,
        "captions.ass"
    )

    header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: 1080
PlayResY: 1920
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,DejaVu Sans,{int(font_size)},{primary},{secondary},&H80000000,&H80000000,{bold},0,0,0,100,100,0,0,1,4,1,5,40,40,40,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""

    events = []

    words_per_line = int(
        words_per_line
    )

    for start_index in range(
        0,
        len(words),
        words_per_line
    ):

        group = words[
            start_index:
            start_index + words_per_line
        ]

        for active_index, active in enumerate(
            group
        ):

            start_time = active["start"]

            if active_index + 1 < len(group):

                end_time = group[
                    active_index + 1
                ]["start"]

            else:

                end_time = group[-1]["end"]

            display_words = []

            for index, item in enumerate(
                group
            ):

                word = (
                    item["word"]
                    .replace("{", "")
                    .replace("}", "")
                    .replace("\\", "")
                )

                if (
                    caption_style == "Karaoke"
                    and index == active_index
                ):

                    display_words.append(
                        r"{\c&H00FFFF&}"
                        + word
                        + r"{\c&HFFFFFF&}"
                    )

                else:

                    display_words.append(
                        word
                    )

            caption = " ".join(
                display_words
            )

            events.append(
                "Dialogue: 0,"
                f"{ass_time(start_time)},"
                f"{ass_time(end_time)},"
                "Default,,0,0,0,,"
                f"{{\\pos(540,{y_position})}}"
                f"{caption}"
            )

    with open(
        ass_file,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(header)
        f.write(
            "\n".join(events)
        )

    print(
        f"✓ {len(words)} words captioned"
    )

    return ass_file


# ============================================================
# IMAGE ANIMATION
# ============================================================

def create_animated_image(
    image,
    duration,
    index
):

    output = os.path.join(
        RUNTIME_TEMP_DIR,
        f"image_{index:03d}.mp4"
    )

    if index % 2 == 0:

        zoom_filter = (
            "zoompan="
            "z='min(zoom+0.0008,1.12)':"
            "x='iw/2-(iw/zoom/2)':"
            "y='ih/2-(ih/zoom/2)':"
            "d=1:"
            "s=1080x1920:"
            "fps=30"
        )

    else:

        zoom_filter = (
            "zoompan="
            "z='if(lte(zoom,1.001),"
            "1.12,"
            "max(zoom-0.0008,1.0))':"
            "x='iw/2-(iw/zoom/2)':"
            "y='ih/2-(ih/zoom/2)':"
            "d=1:"
            "s=1080x1920:"
            "fps=30"
        )

    run_cmd([
        "ffmpeg",
        "-y",
        "-loop",
        "1",
        "-i",
        image,
        "-vf",
        (
            "scale=1080:1920:"
            "force_original_aspect_ratio=increase,"
            "crop=1080:1920,"
            + zoom_filter
        ),
        "-t",
        str(duration),
        "-an",
        "-c:v",
        "libx264",
        "-preset",
        "veryfast",
        "-crf",
        "20",
        "-pix_fmt",
        "yuv420p",
        "-r",
        "30",
        output
    ])

    return output


# ============================================================
# IMAGE SEQUENCE
# ============================================================

def create_image_sequence(
    images,
    narration_duration,
    transition
):

    if len(images) == 1:

        return create_animated_image(
            images[0],
            narration_duration,
            0
        )

    transition_duration = 0.45

    image_duration = (
        narration_duration
        + (
            (len(images) - 1)
            * transition_duration
        )
    ) / len(images)

    print("\n🎬 IMAGE ANIMATION")

    clips = []

    for index, image in enumerate(
        images
    ):

        clip = create_animated_image(
            image,
            image_duration,
            index
        )

        clips.append(clip)

        print(
            f"  ✓ Image "
            f"{index + 1}/{len(images)}"
        )

    current = clips[0]

    transition_name = (
        transition
        if transition != "Auto"
        else "fade"
    )

    for index in range(
        1,
        len(clips)
    ):

        next_clip = clips[index]

        output = os.path.join(
            RUNTIME_TEMP_DIR,
            f"transition_{index:03d}.mp4"
        )

        current_duration = get_duration(
            current
        )

        offset = max(
            0,
            current_duration
            - transition_duration
        )

        filter_complex = (
            f"[0:v][1:v]"
            "xfade="
            f"transition={transition_name}:"
            f"duration={transition_duration}:"
            f"offset={offset},"
            "format=yuv420p[v]"
        )

        run_cmd([
            "ffmpeg",
            "-y",
            "-i",
            current,
            "-i",
            next_clip,
            "-filter_complex",
            filter_complex,
            "-map",
            "[v]",
            "-an",
            "-c:v",
            "libx264",
            "-preset",
            "veryfast",
            "-crf",
            "20",
            "-pix_fmt",
            "yuv420p",
            "-r",
            "30",
            output
        ])

        current = output

    return current


# ============================================================
# HOOK STYLES
# ============================================================

def build_hook(
    hook_style,
    hook_text,
    hook_subtext
):

    if hook_style == "Breaking News":

        return (
            hook_text,
            hook_subtext
        )

    if hook_style == "Question":

        return (
            f"IS THIS THE NEXT BIG THING?",
            hook_text
        )

    if hook_style == "Bold Statement":

        return (
            f"{hook_text} IS READY.",
            hook_subtext
        )

    if hook_style == "Debate":

        return (
            f"LET'S TALK ABOUT {hook_text}",
            hook_subtext
        )

    if hook_style == "Game Day":

        return (
            f"GAMECOCKS FANS, WATCH THIS",
            hook_text
        )

    return (
        hook_text,
        hook_subtext
    )


# ============================================================
# HOOK VIDEO
# ============================================================

def create_hook(
    hook_text,
    hook_subtext,
    duration
):

    hook_file = os.path.join(
        RUNTIME_TEMP_DIR,
        "hook.mp4"
    )

    safe_main = (
        hook_text
        .replace("\\", "")
        .replace(":", "\\:")
        .replace("'", "\\'")
    )

    safe_sub = (
        hook_subtext
        .replace("\\", "")
        .replace(":", "\\:")
        .replace("'", "\\'")
    )

    draw = (
        "drawtext="
        "fontfile=/usr/share/fonts/"
        "truetype/dejavu/"
        "DejaVuSans-Bold.ttf:"
        f"text='{safe_main}':"
        "fontcolor=white:"
        "fontsize=72:"
        "x=(w-text_w)/2:"
        "y=(h-text_h)/2-70:"
        "box=1:"
        "boxcolor=black@0.55:"
        "boxborderw=22,"
        "drawtext="
        "fontfile=/usr/share/fonts/"
        "truetype/dejavu/"
        "DejaVuSans-Bold.ttf:"
        f"text='{safe_sub}':"
        "fontcolor=white:"
        "fontsize=42:"
        "x=(w-text_w)/2:"
        "y=(h-text_h)/2+35:"
        "box=1:"
        "boxcolor=black@0.55:"
        "boxborderw=16"
    )

    run_cmd([
        "ffmpeg",
        "-y",
        "-f",
        "lavfi",
        "-i",
        f"color=c=black:"
        f"s=1080x1920:"
        f"d={duration}:r=30",
        "-vf",
        draw,
        "-an",
        "-c:v",
        "libx264",
        "-preset",
        "veryfast",
        "-crf",
        "20",
        "-pix_fmt",
        "yuv420p",
        "-r",
        "30",
        hook_file
    ])

    return hook_file


# ============================================================
# AUDIO
# ============================================================

def create_final_audio(
    narration,
    music,
    hook,
    hook_duration
):

    final_audio = os.path.join(
        RUNTIME_TEMP_DIR,
        "final_audio.wav"
    )

    delay_ms = (
        int(float(hook_duration) * 1000)
        if hook else 0
    )

    if music:

        music_runtime = os.path.join(
            RUNTIME_TEMP_DIR,
            "music.mp3"
        )

        shutil.copy2(
            music,
            music_runtime
        )

        voice_filter = (
            f"adelay={delay_ms}|{delay_ms},"
            if hook
            else ""
        )

        voice_filter += (
            "highpass=f=70,"
            "lowpass=f=14000,"
            "acompressor="
            "threshold=0.08:"
            "ratio=2.5:"
            "attack=15:"
            "release=180"
        )

        run_cmd([
            "ffmpeg",
            "-y",
            "-i",
            narration,
            "-stream_loop",
            "-1",
            "-i",
            music_runtime,
            "-filter_complex",
            (
                f"[0:a]{voice_filter}[voice];"
                "[1:a]"
                "volume=0.12,"
                "highpass=f=40,"
                "lowpass=f=12000"
                "[music];"
                "[music][voice]"
                "sidechaincompress="
                "threshold=0.02:"
                "ratio=8:"
                "attack=20:"
                "release=350"
                "[ducked];"
                "[voice][ducked]"
                "amix=inputs=2:"
                "duration=first:"
                "dropout_transition=2,"
                "loudnorm="
                "I=-16:"
                "LRA=7:"
                "TP=-1.5"
                "[out]"
            ),
            "-map",
            "[out]",
            "-ar",
            "48000",
            "-ac",
            "2",
            "-c:a",
            "pcm_s16le",
            final_audio
        ])

    else:

        audio_filter = (
            f"adelay={delay_ms}|{delay_ms},"
            if hook
            else ""
        )

        audio_filter += (
            "loudnorm="
            "I=-16:"
            "LRA=7:"
            "TP=-1.5"
        )

        run_cmd([
            "ffmpeg",
            "-y",
            "-i",
            narration,
            "-af",
            audio_filter,
            "-ar",
            "48000",
            "-ac",
            "2",
            "-c:a",
            "pcm_s16le",
            final_audio
        ])

    return final_audio


# ============================================================
# GENERATE VIDEO
# ============================================================

def generate_video(
    script,
    media_files,
    captions,
    hook,
    hook_style,
    hook_text,
    hook_subtext,
    hook_duration,
    speech_rate,
    voice_name,
    pacing,
    font_size,
    words_per_line,
    caption_position,
    caption_style,
    music_file,
    transition
):

    if not script or not script.strip():

        raise gr.Error(
            "Please enter a script."
        )

    if not media_files:

        raise gr.Error(
            "Please upload at least one image or video."
        )

    # --------------------------------------------------------
    # CLEAN RUNTIME
    # --------------------------------------------------------

    for filename in os.listdir(
        RUNTIME_MEDIA_DIR
    ):

        path = os.path.join(
            RUNTIME_MEDIA_DIR,
            filename
        )

        if os.path.isfile(path):

            os.remove(path)

    copied_media = []

    for file in media_files:

        source = (
            file.get("path")
            if isinstance(file, dict)
            else file
        )

        if not source:
            continue

        destination = os.path.join(
            RUNTIME_MEDIA_DIR,
            os.path.basename(source)
        )

        shutil.copy2(
            source,
            destination
        )

        copied_media.append(
            destination
        )

        shutil.copy2(
            source,
            os.path.join(
                PERSISTENT_MEDIA_DIR,
                os.path.basename(source)
            )
        )

    images = [
        f for f in copied_media
        if f.lower().endswith(
            (
                ".jpg",
                ".jpeg",
                ".png",
                ".webp"
            )
        )
    ]

    videos = [
        f for f in copied_media
        if f.lower().endswith(
            (
                ".mp4",
                ".mov",
                ".mkv",
                ".avi",
                ".webm"
            )
        )
    ]

    if videos and images:

        raise gr.Error(
            "Upload either images OR ONE video."
        )

    if len(videos) > 1:

        raise gr.Error(
            "Upload only ONE video."
        )

    # --------------------------------------------------------
    # NARRATION
    # --------------------------------------------------------

    narration = create_narration(
        script,
        speech_rate,
        voice_name,
        pacing
    )

    narration_duration = get_duration(
        narration
    )

    # --------------------------------------------------------
    # VISUAL
    # --------------------------------------------------------

    if videos:

        visual = os.path.join(
            RUNTIME_TEMP_DIR,
            "visual.mp4"
        )

        run_cmd([
            "ffmpeg",
            "-y",
            "-i",
            videos[0],
            "-vf",
            (
                "scale=1080:1920:"
                "force_original_aspect_ratio=increase,"
                "crop=1080:1920"
            ),
            "-an",
            "-t",
            str(narration_duration),
            "-c:v",
            "libx264",
            "-preset",
            "veryfast",
            "-crf",
            "20",
            "-pix_fmt",
            "yuv420p",
            "-r",
            "30",
            visual
        ])

    else:

        visual = create_image_sequence(
            images,
            narration_duration,
            transition
        )

    # --------------------------------------------------------
    # HOOK
    # --------------------------------------------------------

    final_visual = visual

    if hook:

        final_hook_text, final_hook_subtext = (
            build_hook(
                hook_style,
                hook_text,
                hook_subtext
            )
        )

        hook_file = create_hook(
            final_hook_text,
            final_hook_subtext,
            hook_duration
        )

        combined = os.path.join(
            RUNTIME_TEMP_DIR,
            "visual_with_hook.mp4"
        )

        concat_file = os.path.join(
            RUNTIME_TEMP_DIR,
            "hook_concat.txt"
        )

        with open(
            concat_file,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                f"file '{hook_file}'\n"
            )

            f.write(
                f"file '{visual}'\n"
            )

        run_cmd([
            "ffmpeg",
            "-y",
            "-f",
            "concat",
            "-safe",
            "0",
            "-i",
            concat_file,
            "-an",
            "-c:v",
            "libx264",
            "-preset",
            "veryfast",
            "-crf",
            "20",
            "-pix_fmt",
            "yuv420p",
            "-r",
            "30",
            combined
        ])

        final_visual = combined

    # --------------------------------------------------------
    # AUDIO
    # --------------------------------------------------------

    final_audio = create_final_audio(
        narration,
        music_file,
        hook,
        hook_duration
    )

    # --------------------------------------------------------
    # CAPTIONS
    # --------------------------------------------------------

    caption_file = None

    if captions:

        caption_file = create_captions(
            narration,
            font_size,
            words_per_line,
            caption_position,
            caption_style
        )

        if hook:

            synced_file = os.path.join(
                RUNTIME_TEMP_DIR,
                "captions_synced.ass"
            )

            with open(
                caption_file,
                "r",
                encoding="utf-8"
            ) as f:

                lines = f.readlines()

            shifted = []

            for line in lines:

                if not line.startswith(
                    "Dialogue:"
                ):

                    shifted.append(line)
                    continue

                parts = line.rstrip(
                    "\n"
                ).split(",", 9)

                if len(parts) < 10:

                    shifted.append(line)
                    continue

                def shift_time(value):

                    h, m, s = value.split(":")

                    total = (
                        int(h) * 3600
                        + int(m) * 60
                        + float(s)
                        + float(hook_duration)
                    )

                    return ass_time(total)

                parts[1] = shift_time(
                    parts[1]
                )

                parts[2] = shift_time(
                    parts[2]
                )

                shifted.append(
                    ",".join(parts) + "\n"
                )

            with open(
                synced_file,
                "w",
                encoding="utf-8"
            ) as f:

                f.writelines(shifted)

            caption_file = synced_file

    # --------------------------------------------------------
    # FINAL RENDER
    # --------------------------------------------------------

    output = os.path.join(
        PERSISTENT_OUTPUT_DIR,
        "gamecocks_story.mp4"
    )

    command = [
        "ffmpeg",
        "-y",
        "-i",
        final_visual,
        "-i",
        final_audio
    ]

    if caption_file:

        caption_path = (
            caption_file
            .replace("\\", "/")
            .replace(":", "\\:")
        )

        command.extend([
            "-vf",
            f"ass='{caption_path}'"
        ])

    command.extend([
        "-map",
        "0:v",
        "-map",
        "1:a",
        "-c:v",
        "libx264",
        "-preset",
        "veryfast",
        "-crf",
        "20",
        "-pix_fmt",
        "yuv420p",
        "-c:a",
        "aac",
        "-b:a",
        "192k",
        "-ar",
        "48000",
        "-shortest",
        output
    ])

    run_cmd(command)

    if not os.path.exists(output):

        raise RuntimeError(
            "Final video was not created."
        )

    print("\n========================================")
    print("✅ VIDEO COMPLETE")
    print("========================================")

    print("Saved:", output)

    return output


# ============================================================
# PROCESS
# ============================================================

def process(
    script,
    media,
    voice_name,
    speech_rate,
    pacing,
    captions,
    font_size,
    words_per_line,
    caption_position,
    caption_style,
    hook,
    hook_style,
    hook_text,
    hook_subtext,
    hook_duration,
    music_enabled,
    music,
    transition
):

    music_path = None

    if music_enabled and music:

        music_path = (
            music.get("path")
            if isinstance(music, dict)
            else music
        )

        if music_path:

            persistent_music = os.path.join(
                PERSISTENT_MUSIC_DIR,
                os.path.basename(music_path)
            )

            shutil.copy2(
                music_path,
                persistent_music
            )

    return generate_video(
        script,
        media,
        captions,
        hook,
        hook_style,
        hook_text,
        hook_subtext,
        hook_duration,
        speech_rate,
        voice_name,
        pacing,
        font_size,
        words_per_line,
        caption_position,
        caption_style,
        music_path,
        transition
    )


# ============================================================
# GRADIO UI
# ============================================================

with gr.Blocks(
    title="Gamecocks Video Maker"
) as app:

    gr.Markdown(
        """
        # 🏀 GAMECOCKS VIDEO MAKER
        Create a vertical 9:16 sports video.
        """
    )

    script = gr.Textbox(
        label="SCRIPT",
        lines=8,
        placeholder="Paste your finished sports script here..."
    )

    media = gr.File(
        label="MEDIA — Images or ONE Video",
        file_count="multiple",
        type="filepath"
    )

    voice = gr.Dropdown(
        choices=list(PIPER_VOICES.keys()),
        value=list(PIPER_VOICES.keys())[0],
        label="NARRATOR — Voice"
    )

    speech_rate = gr.Slider(
        minimum=0.85,
        maximum=1.15,
        value=1.00,
        step=0.01,
        label="SPEED"
    )

    pacing = gr.Dropdown(
        choices=[
            "Breaking News",
            "Normal Sports Story",
            "High Energy",
            "Analysis"
        ],
        value="Normal Sports Story",
        label="PACING"
    )

    with gr.Row():

        captions = gr.Checkbox(
            value=True,
            label="CAPTIONS"
        )

        hook = gr.Checkbox(
            value=True,
            label="HOOK"
        )

        music_enabled = gr.Checkbox(
            value=True,
            label="MUSIC"
        )

    with gr.Row():

        font_size = gr.Slider(
            minimum=36,
            maximum=90,
            value=58,
            step=2,
            label="Caption Font Size"
        )

        words_per_line = gr.Slider(
            minimum=2,
            maximum=8,
            value=5,
            step=1,
            label="Words Per Line"
        )

    with gr.Row():

        caption_position = gr.Dropdown(
            choices=[
                "Top",
                "Center",
                "Bottom"
            ],
            value="Bottom",
            label="Caption Position"
        )

        caption_style = gr.Dropdown(
            choices=[
                "Karaoke",
                "Clean",
                "Bold"
            ],
            value="Karaoke",
            label="Caption Style"
        )

    hook_style = gr.Dropdown(
        choices=[
            "Breaking News",
            "Question",
            "Bold Statement",
            "Debate",
            "Game Day"
        ],
        value="Breaking News",
        label="HOOK STYLE"
    )

    hook_text = gr.Textbox(
        value="BROOKE QUARLES DANIELS",
        label="Hook Main Text"
    )

    hook_subtext = gr.Textbox(
        value="IS PUTTING IN THE WORK",
        label="Hook Subtext"
    )

    hook_duration = gr.Slider(
        minimum=0.5,
        maximum=3.0,
        value=1.5,
        step=0.1,
        label="Hook Duration"
    )

    music = gr.File(
        label="Background Music",
        file_count="single",
        type="filepath"
    )

    transition = gr.Dropdown(
        choices=[
            "Auto",
            "fade",
            "slideleft",
            "slideright",
            "wipeleft",
            "wiperight"
        ],
        value="Auto",
        label="TRANSITION"
    )

    generate = gr.Button(
        "🎬 GENERATE VIDEO",
        variant="primary"
    )

    output = gr.Video(
        label="FINAL VIDEO"
    )

    generate.click(
        fn=process,
        inputs=[
            script,
            media,
            voice,
            speech_rate,
            pacing,
            captions,
            font_size,
            words_per_line,
            caption_position,
            caption_style,
            hook,
            hook_style,
            hook_text,
            hook_subtext,
            hook_duration,
            music_enabled,
            music,
            transition
        ],
        outputs=output
    )


print("========================================")
print("🚀 GRADIO APP READY")
print("========================================")

app.launch(
    share=True
)

🚀 GRADIO APP READY
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f6d7f832ebebf542ed.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:

# ============================================================
# CELL 22 — FIX GRADIO
# ============================================================

import sys
import subprocess

print("🔧 Installing stable Gradio version...")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--force-reinstall",
    "gradio==5.49.1"
])

print("\n✅ GRADIO INSTALL COMPLETE")
print("Now RE-RUN existing Cell 21.")

🔧 Installing stable Gradio version...

✅ GRADIO INSTALL COMPLETE
Now RE-RUN existing Cell 21.
